# Day 3：卷积、LeNet 与结果分析

前置：能够独立解释 Day 2 的训练循环；否则先巩固 MLP。本日不要求残差网络。

In [ ]:
from pathlib import Path
import os, sys, subprocess
root = Path.cwd()
if not (root / "labs").exists():
    root = root.parent
assert (root / "labs").exists(), "请从仓库或 notebooks 目录打开"
os.chdir(root)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
import torch
print("Python:", sys.executable, "CUDA:", torch.cuda.is_available())
def run_lab(module, *args):
    subprocess.run([sys.executable, "-m", "labs." + module, *args], check=True)


先读 [Day 3](../lessons/day03.md)。先手算 shape，再看实际输出。

In [ ]:
from labs.vision import make_model
x = torch.zeros(2, 1, 28, 28)
model = make_model("lenet")
for layer in model:
    x = layer(x)
    print(type(layer).__name__, tuple(x.shape))


练习：4×4 输入和 2×2 核，手算卷积输出，再用 `conv2d` 验证。

In [ ]:
# 在这里写手算与验证。


In [ ]:
run_lab("vision", "--model", "lenet", "--train-size", "6000", "--epochs", "10", "--lr", "0.003", "--seed", "42")


In [ ]:
# 只加载本仓库自己产生的权重；检查保存/加载是否保持预测。
paths = sorted(Path("runs").glob("*vision-lenet/model.pt"))
assert paths
saved = torch.load(paths[-1], map_location="cpu", weights_only=True)
first = make_model(saved["model"]).eval()
first.load_state_dict(saved["state_dict"])
second = make_model(saved["model"]).eval()
second.load_state_dict(first.state_dict())
x = torch.randn(4, 1, 28, 28)
with torch.no_grad():
    torch.testing.assert_close(first(x), second(x))
print("Reload consistency PASS")


练习：在 6k 验证集上找 16 个错例、画图并解释。使用固定 split_seed=42，不看测试集。

In [ ]:
# 在这里收集验证集错例；从 labs.vision 读取划分方法。


三日总结：不看参考代码，解释一轮训练、数据划分和模型差异。LeNet 与 MLP 学习率不同，比较不能隔离架构的因果作用。完成 [阶段检查](../notes/FINAL_CHECK.md) 第 1–4、6 题。

我的解释与仍待解决的问题：